## RainPro-8 (Architecture)

Demo notebook for using the full RainPro-8 architecture combining multi-source data with different spatial coverage, spatial resolution, and timesteps.  
For details regarding training, we refer to the README with the SEVIR benchmark.

<img src="images/arch.png" width=700 />

<img src="images/sources.png" width=700 />

In [1]:
import torch
from rainpro.network.rainpro8 import RainPro, StackTimeAndChannels, EvalRequest

### Data Sources

In [2]:
sources = {
    "target_2km": {"resolution": 2, "size_km": 512, "timesteps": 48, "channels": 1},
    "radar_4km": {"resolution": 4, "size_km": 1024, "timesteps": 7, "channels": 1},
    "radar_8km": {"resolution": 8, "size_km": 1536, "timesteps": 1, "channels": 1},
    "satellite_8km": {"resolution": 8, "size_km": 1536, "timesteps": 5, "channels": 11},
    "gfs_16km": {"resolution": 16, "size_km": 1536, "timesteps": 1, "channels": 122},
    "gfs_forecast_16km": {"resolution": 16, "size_km": 1536, "timesteps": 8, "channels": 1},
    "xyz_4km": {"resolution": 4, "size_km": 1024, "timesteps": 1, "channels": 3},
    "minute_4km": {"resolution": 4, "size_km": 1024, "timesteps": 1, "channels": 1},
}

In [3]:
# Add pixel sizes
for s in sources.values():
    s["size_px"] = s["size_km"] // s["resolution"]

In [4]:
# Compute input channels for each resolution
res_dim = {4: 0, 8: 0, 16: 0}

for s in sources.values():
    r = s["resolution"]
    if r in res_dim:
        res_dim[r] += s["timesteps"] * s["channels"]

print("Input channels for each resolution (stacked time and channels)")
print(res_dim)

Input channels for each resolution (stacked time and channels)
{4: 11, 8: 56, 16: 130}


In [5]:
# Compute context sizes for padding and cropping
target_size_2km = sources["target_2km"]["size_px"]
input_size_4km = sources["radar_4km"]["size_px"]
context_size_4km = (input_size_4km - target_size_2km // 2) // 2
input_size_8km = sources["radar_8km"]["size_px"]
context_size_8km = (input_size_8km - target_size_2km // 4) // 2

print("Context with respect to target region on each side:")
print(f" Smaller region (4km inputs): {context_size_4km}px")
print(f" Larger region (8km inputs): {context_size_8km}px")
print(f" Larger region (16km inputs): {context_size_8km // 2}px")

Context with respect to target region on each side:
 Smaller region (4km inputs): 64px
 Larger region (8km inputs): 64px
 Larger region (16km inputs): 32px


### RainPro-8

In [6]:
model = RainPro(
    T_out=sources["target_2km"]["timesteps"],
    in_dims=(res_dim[4], res_dim[8], res_dim[16]),
    context_size_4km=context_size_4km,
    context_size_8km=context_size_8km,
)

In [7]:
B = 2
inputs = {
    name: torch.randn(
        B,
        s["timesteps"],
        s["channels"],
        s["size_px"],
        s["size_px"],
    )
    for name, s in sources.items()
}

In [8]:
stack_time_and_channels = StackTimeAndChannels()


def stack_sources(tensors: list[torch.Tensor]) -> torch.Tensor:
    stacked = [stack_time_and_channels(x) for x in tensors]
    tensor = torch.cat(stacked, dim=1)
    return tensor


x_4km = stack_sources([inputs[s] for s in sources if sources[s]["resolution"] == 4])
x_8km = stack_sources([inputs[s] for s in sources if sources[s]["resolution"] == 8])
x_16km = stack_sources([inputs[s] for s in sources if sources[s]["resolution"] == 16])

In [9]:
eval_request = EvalRequest(need_forecast=True, need_probs=True)
eval_output = model.predict(
    x_4km,
    x_8km,
    x_16km,
    target=None,
    eval_request=eval_request,
)

print("Probabilities:", eval_output.probs.shape)
print("Forecast:", eval_output.forecast.shape)

Probabilities: torch.Size([2, 48, 9, 256, 256])
Forecast: torch.Size([2, 48, 1, 256, 256])


In [10]:
pred = model.forward(x_4km, x_8km, x_16km)
target = inputs["target_2km"]
loss = model.criterion(pred, target)

print(loss)

tensor(0.7389, grad_fn=<MeanBackward0>)
